# 04 · Sentiment Analysis & NLP Deep-Dive

The NLP module adds a **market-mood signal** on top of fundamental valuation.
This notebook walks through how it works — from raw headlines to an actionable composite score.

---

**What you will learn:**

1. How the three-tier NLP backend scores financial headlines
2. How uncertainty and guidance revision signals are extracted
3. How to fetch and interpret live sentiment for any stock
4. How to scan a universe in parallel and rank by sentiment momentum
5. How to combine valuation + sentiment into one composite signal

> **Key design principle**: sentiment adjusts *confidence* only — never the fair value.
> Noisy short-term news should not move a 10-year DCF by even one dollar.

In [ ]:
import sys
import pathlib

sys.path.insert(0, str(pathlib.Path().resolve().parent))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from concurrent.futures import ThreadPoolExecutor, as_completed

from fairprice.data import FinancialsClient, MarketClient, MacroClient
from fairprice.nlp import SentimentClient
from fairprice.nlp.sentiment import (
    active_backend, score_texts, _keyword,
    detect_uncertainty, detect_guidance_revision,
)
import fairprice.valuation as valuation_engine

fin = FinancialsClient()
mkt = MarketClient()
mac = MacroClient()
nlp = SentimentClient()

print(f"Active NLP backend : {active_backend().upper()}")
print("All clients ready.")
print()
print("Backend hierarchy (auto-selected at import):")
print("  FinBERT  — best quality, needs GPU / transformers+torch")
print("  VADER    — good quality, no GPU, install: pip install vaderSentiment")
print("  Keyword  — always available, uses Loughran-McDonald financial lexicon")

## 1 · NLP Backend — Scoring Financial Headlines

FairPrice uses three NLP tiers, auto-selected at startup based on what is installed:

| Tier | Package | Quality | Speed | GPU? |
|------|---------|---------|-------|------|
| **FinBERT** | `transformers + torch` | ★★★★★ | Slow | Optional |
| **VADER** | `vaderSentiment` | ★★★☆☆ | Fast | No |
| **Keyword** | built-in lexicon | ★★☆☆☆ | Instant | No |

The **keyword backend** uses the Loughran-McDonald financial word lists — a domain-specific
lexicon that knows "miss" and "warning" are negative while "beat" and "raised" are positive.
General-purpose lexicons (like VADER's default) frequently mis-score financial text.

Below: score seven example headlines and see what each tier picks up.

In [ ]:
DEMO_HEADLINES = [
    "Apple beats Q3 estimates; raises full-year guidance above consensus",
    "Company misses revenue expectations; issues profit warning for Q4",
    "Fed holds rates steady; markets remain uncertain about the trajectory",
    "Record free cash flow and strong operating margins drive the beat",
    "Supply chain disruptions may weigh on margins next quarter",
    "Analyst upgrades stock to Buy citing compelling valuation support",
    "CEO departure raises governance concerns among institutional investors",
]

# Score with whichever backend is active
results = score_texts(DEMO_HEADLINES)

# Always also show pure keyword scores for transparency
kw_results = _keyword(DEMO_HEADLINES)

print(f"Backend in use: {active_backend().upper()}\n")
print(f"{'Sig':3}  {'Act Pos':>7}  {'Act Neg':>7}  {'KW Pos':>6}  {'KW Neg':>6}  Headline")
print("─" * 100)
for h, r, kw in zip(DEMO_HEADLINES, results, kw_results):
    net = r["positive"] - r["negative"]
    sig = "📈" if net > 0.15 else "📉" if net < -0.15 else "➖"
    print(
        f"{sig}   {r['positive']:>6.0%}  {r['negative']:>6.0%}  "
        f"{kw['positive']:>5.0%}  {kw['negative']:>5.0%}  "
        f"{h[:60]}"
    )

In [ ]:
# Visualise keyword scores as stacked horizontal bars
labels = [h[:50] + "…" if len(h) > 50 else h for h in DEMO_HEADLINES]

fig = go.Figure()
for col, color, name in [
    ("positive", "#4CAF50", "Positive"),
    ("neutral",  "#9E9E9E", "Neutral"),
    ("negative", "#F44336", "Negative"),
]:
    fig.add_trace(go.Bar(
        name=name, x=[r[col] for r in kw_results], y=labels,
        orientation="h", marker_color=color,
        text=[f"{r[col]:.0%}" for r in kw_results],
        textposition="inside", insidetextanchor="middle",
    ))

fig.update_layout(
    barmode="stack",
    title="Keyword Lexicon — Example Financial Headline Scores",
    xaxis_title="Score  (0 – 1)",
    height=400,
    legend=dict(orientation="h", y=1.08),
)
fig.show()

## 2 · Uncertainty & Guidance Revision Detection

Beyond positive/negative polarity, two extra signals are extracted:

**Uncertainty score** (0–1)
: How much hedging / forward ambiguity is in the text. High uncertainty → confidence
  penalty even if overall tone is neutral. Words like *uncertain*, *volatile*,
  *challenging*, *headwinds*, *may*, *could* trigger this.

**Guidance revision**
: Did management *raise*, *lower*, or *reiterate* guidance?
  This is one of the highest signal-to-noise indicators in earnings season —
  it directly predicts near-term analyst estimate revisions.

Precedence rule when multiple signals co-occur: **lowered > reiterated > raised**
(conservative — if bad news is present, take it seriously).

In [ ]:
UNCERTAINTY_EXAMPLES = [
    ("HIGH",   "The outlook remains uncertain amid volatile market conditions and challenging headwinds"),
    ("HIGH",   "Risks include potential regulatory changes, uncertain demand environment, and macro headwinds"),
    ("MEDIUM", "Results were mixed; management cautious about next quarter but maintains annual guidance"),
    ("LOW",    "Apple reported record revenue of $120 billion, raised guidance above consensus"),
    ("LOW",    "Quarterly results beat on every metric; management confident about full-year outlook"),
]

print("Uncertainty score  (0 = crystal-clear … 1 = very ambiguous):\n")
for label, text in UNCERTAINTY_EXAMPLES:
    unc    = detect_uncertainty(text)
    filled = int(unc * 30)
    bar    = "█" * filled + "░" * (30 - filled)
    print(f"  {label:<6}  [{bar}]  {unc:.3f}")
    print(f"          "{text[:80]}"")
    print()

In [ ]:
GUIDANCE_CASES = [
    (["Management raised full-year guidance above analyst consensus"],               "raised"),
    (["Company issued a profit warning and lowered guidance significantly"],          "lowered"),
    (["Management reaffirmed guidance in line with market expectations"],             "reiterated"),
    (["maintained guidance but then lowered guidance significantly"],                 "lowered (beats reiterated)"),
    (["raised guidance last quarter", "now warns of significant challenges ahead"],   "lowered (latest wins)"),
    (["Quarterly results next week", "CEO speaks at industry conference"],            "None — no guidance signal"),
]

print("Guidance revision detection:\n")
for texts, expected in GUIDANCE_CASES:
    detected = detect_guidance_revision(texts)
    marker   = "✓" if str(detected) in expected else "→"
    print(f"  {marker}  Input    : "{texts[0][:70]}"")
    if len(texts) > 1:
        print(f"             + "{texts[1][:70]}"")
    print(f"     Detected : {detected or 'None':<12}  (expected: {expected})")
    print()

## 3 · Live Sentiment — Single Stock Deep-Dive

Fetch live news from all configured sources (yfinance, FinViz, Alpaca News),
score every article, and display a full breakdown with per-headline detail.

> First run: ~3–5 s (live API calls).
> Subsequent runs: instant (1-hour disk cache).

Change `TICKER` below to analyse any stock.

In [ ]:
TICKER = "AAPL"  # ← change to any ticker

print(f"Fetching live sentiment for {TICKER}...")
sent = nlp.get_sentiment(TICKER)

print(f"\n{'='*60}")
print(f"  Sentiment Report: {TICKER}")
print(f"{'='*60}")
print(f"  Backend           : {sent.backend.upper()}")
print(f"  Sources used      : {', '.join(sent.sources) or 'none (check .env API keys)'}")
print(f"  yfinance + FinViz : {sent.n_articles} articles")
print(f"  Alpaca News       : {sent.n_alpaca_articles} articles")
total = sent.n_articles + sent.n_alpaca_articles
print(f"  Total             : {total}")
print()
print(f"  Sentiment score   : {sent.sentiment_score:+.4f}  (−1 very bearish … +1 very bullish)")
print(f"  Positive          : {sent.positive_ratio:.1%}")
print(f"  Negative          : {sent.negative_ratio:.1%}")
print(f"  Neutral           : {sent.neutral_ratio:.1%}")
print(f"  Uncertainty       : {sent.uncertainty_score:.4f}")
if sent.guidance_revision:
    print(f"  Guidance revision : {sent.guidance_revision.upper()}")
if sent.analyst_consensus:
    print(f"  Analyst consensus : {sent.analyst_consensus.upper()}")

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "indicator"}, {"type": "bar"}]],
    subplot_titles=["Sentiment Gauge", "Pos / Neutral / Neg Breakdown"],
)

gauge_color = (
    "#4CAF50" if sent.sentiment_score > 0.1
    else "#F44336" if sent.sentiment_score < -0.1
    else "#9E9E9E"
)
fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=sent.sentiment_score,
        gauge=dict(
            axis=dict(range=[-1, 1], tickvals=[-1, -0.3, 0, 0.3, 1]),
            bar=dict(color=gauge_color),
            steps=[
                dict(range=[-1.0, -0.3], color="#FFCDD2"),
                dict(range=[-0.3,  0.3], color="#FFF9C4"),
                dict(range=[ 0.3,  1.0], color="#C8E6C9"),
            ],
            threshold=dict(line=dict(color="black", width=2), value=0),
        ),
        title=dict(text=TICKER),
        number=dict(valueformat="+.3f"),
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Bar(
        x=["Positive", "Neutral", "Negative"],
        y=[sent.positive_ratio, sent.neutral_ratio, sent.negative_ratio],
        marker_color=["#4CAF50", "#9E9E9E", "#F44336"],
        text=[f"{v:.1%}" for v in [sent.positive_ratio, sent.neutral_ratio, sent.negative_ratio]],
        textposition="outside",
        showlegend=False,
    ),
    row=1, col=2,
)

fig.update_layout(height=370, title_text=f"{TICKER} — Sentiment Detail  [{sent.backend}]")
fig.update_yaxes(range=[0, 1.2], row=1, col=2)
fig.show()

# Uncertainty bar
unc    = sent.uncertainty_score
filled = int(unc * 30)
bar    = "█" * filled + "░" * (30 - filled)
level  = "HIGH" if unc > 0.5 else "MEDIUM" if unc > 0.2 else "LOW"
print(f"\nUncertainty: [{bar}] {unc:.3f}  ({level})")
if unc > 0.5:
    print("  → High uncertainty penalises confidence score (× 0.90 when also bearish).")

In [ ]:
if sent.top_headlines:
    texts  = sent.top_headlines
    scores = score_texts(texts)
    uncs   = [detect_uncertainty(t) for t in texts]

    # Diverging bar chart — positive right, negative left
    df_hl = pd.DataFrame([
        {
            "Headline": h[:80],
            "Positive": s["positive"],
            "Negative": s["negative"],
            "Net":      s["positive"] - s["negative"],
            "Unc":      u,
        }
        for h, s, u in zip(texts, scores, uncs)
    ])

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="Positive", y=df_hl["Headline"], x=df_hl["Positive"],
        orientation="h", marker_color="#4CAF50",
        text=[f"+{v:.0%}" for v in df_hl["Positive"]], textposition="outside",
    ))
    fig.add_trace(go.Bar(
        name="Negative", y=df_hl["Headline"], x=-df_hl["Negative"],
        orientation="h", marker_color="#F44336",
        text=[f"-{v:.0%}" for v in df_hl["Negative"]], textposition="outside",
    ))
    fig.add_vline(x=0, line_color="black", line_width=1)
    fig.update_layout(
        barmode="relative",
        title=f"{TICKER} — Per-headline Sentiment  (← negative | positive →)",
        xaxis_title="Score",
        height=max(300, len(texts) * 52 + 120),
        bargap=0.35,
        legend=dict(orientation="h", y=1.05),
    )
    fig.show()

    print("\nDetailed scores:")
    print(f"{'':3}  {'Pos':>5}  {'Neg':>5}  {'Net':>6}  {'Unc':>5}  Headline")
    print("─" * 92)
    for _, row in df_hl.iterrows():
        sig = "📈" if row["Net"] > 0.15 else "📉" if row["Net"] < -0.15 else "➖"
        print(f"{sig}   {row['Positive']:>4.0%}  {row['Negative']:>4.0%}  "
              f"{row['Net']:>+5.2f}  {row['Unc']:>4.2f}  {row['Headline'][:68]}")
else:
    print("No live headlines available.")
    print("Configure ALPACA_API_KEY + ALPACA_SECRET in .env for the richest news feed.")

## 4 · Sentiment Sweep — Universe Comparison

Fetch sentiment for 10 tickers across sectors in parallel.

> **First run**: ~5–15 s (live API, rate-limited).
> **Subsequent runs**: instant from 1-hour disk cache.

Insight: the *relative* ranking matters more than absolute scores.
A stock with sentiment score +0.3 in a universe where most are at −0.1 has
meaningful positive momentum compared to peers — even if +0.3 sounds modest.

In [ ]:
SWEEP = {
    "AAPL": "Technology", "MSFT": "Technology",
    "NVDA": "Technology", "META": "Technology", "IBM": "Technology",
    "PFE":  "Healthcare", "MDT": "Healthcare",
    "BAC":  "Financials",
    "MCD":  "Consumer",
    "XOM":  "Energy",
}

print(f"Fetching sentiment for {len(SWEEP)} tickers in parallel  [{active_backend()} backend]...\n")

sent_map: dict = {}

def _fetch_sent(t):
    return t, nlp.get_sentiment(t)

with ThreadPoolExecutor(max_workers=5) as pool:
    futures = {pool.submit(_fetch_sent, t): t for t in SWEEP}
    for fut in as_completed(futures):
        t, s = fut.result()
        sent_map[t] = s
        b_pos = "▓" * int(s.positive_ratio * 10)
        b_neg = "▓" * int(s.negative_ratio * 10)
        print(
            f"  {t:<6}  score={s.sentiment_score:+.3f}  "
            f"pos={s.positive_ratio:.0%} [{b_pos:<10}]  "
            f"neg={s.negative_ratio:.0%} [{b_neg:<10}]  "
            f"unc={s.uncertainty_score:.2f}  n={s.n_articles + s.n_alpaca_articles}"
        )

print("\nAll done.")

In [ ]:
rows_sw = []
for t, sector in SWEEP.items():
    s = sent_map[t]
    rows_sw.append({
        "Ticker":      t,
        "Sector":      sector,
        "Sent Score":  s.sentiment_score,
        "Positive":    s.positive_ratio,
        "Neutral":     s.neutral_ratio,
        "Negative":    s.negative_ratio,
        "Uncertainty": s.uncertainty_score,
        "N Articles":  s.n_articles + s.n_alpaca_articles,
    })

df_sw = pd.DataFrame(rows_sw).sort_values("Sent Score", ascending=False)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Sentiment Score Ranking", "Pos / Neutral / Neg Breakdown"],
    column_widths=[0.38, 0.62],
)

fig.add_trace(go.Bar(
    x=df_sw["Sent Score"], y=df_sw["Ticker"], orientation="h",
    marker_color=[
        "#4CAF50" if v > 0.1 else "#F44336" if v < -0.1 else "#9E9E9E"
        for v in df_sw["Sent Score"]
    ],
    text=[f"{v:+.3f}" for v in df_sw["Sent Score"]],
    textposition="outside", showlegend=False,
), row=1, col=1)

for col, color, name in [
    ("Positive", "#4CAF50", "Positive"),
    ("Neutral",  "#9E9E9E", "Neutral"),
    ("Negative", "#F44336", "Negative"),
]:
    fig.add_trace(go.Bar(
        name=name, y=df_sw["Ticker"], x=df_sw[col],
        orientation="h", marker_color=color,
    ), row=1, col=2)

fig.add_vline(x= 0.3, line_dash="dash", line_color="#4CAF50",
              annotation_text="Bullish", row=1, col=1)
fig.add_vline(x=-0.3, line_dash="dash", line_color="#F44336",
              annotation_text="Bearish", row=1, col=1)

fig.update_layout(
    barmode="stack", height=460,
    title="Sentiment Sweep — 10 Tickers",
    legend=dict(orientation="h", y=1.08),
)
fig.show()

# Relative ranking insight
mean_score = df_sw["Sent Score"].mean()
print(f"Universe average sentiment: {mean_score:+.3f}")
print()
above = df_sw[df_sw["Sent Score"] > mean_score + 0.05]
below = df_sw[df_sw["Sent Score"] < mean_score - 0.05]
if not above.empty:
    print(f"Above-average positive momentum : {', '.join(above['Ticker'])}")
if not below.empty:
    print(f"Below-average / negative momentum: {', '.join(below['Ticker'])}")

## 5 · Valuation × Sentiment Quadrant Map

Combining fundamental fair value (margin of safety) with sentiment gives
a **two-dimensional signal** that is more robust than either dimension alone:

| MoS | Sentiment | Quadrant |
|-----|-----------|----------|
| > +15 % | Positive | ★ **STRONG BUY** — cheap *and* tailwinds |
| > +15 % | Negative | ⚡ **VALUE TRAP?** — cheap but news flow is bearish |
| < −15 % | Positive | ⚠ **MOMENTUM PLAY** — expensive but near-term buzz |
| < −15 % | Negative | ▼ **CLEAR AVOID** — expensive *and* headwinds |

This chart uses the **full screener results** saved by notebook 03.
If that file is absent, a mini-universe is fetched inline.

In [ ]:
# ── Load screener data ────────────────────────────────────────────────────────
screener_csv = pathlib.Path("screener_results.csv")

if screener_csv.exists():
    df_screen = pd.read_csv(screener_csv)
    print(f"Loaded {len(df_screen)} tickers from screener_results.csv")
else:
    print("screener_results.csv not found — fetching a 12-ticker mini-universe...")
    MINI = {
        "AAPL":"Technology", "MSFT":"Technology", "META":"Technology", "IBM":"Technology",
        "PFE":"Healthcare",  "MDT":"Healthcare",
        "BAC":"Financials",
        "MCD":"Consumer",    "KO":"Consumer",
        "XOM":"Energy",      "CVX":"Energy",
        "GD":"Defense",
    }
    macro_m = mac.get_macro_data()
    rows_m  = []
    for t, sector in MINI.items():
        try:
            stmts_m  = fin.get_statements(t)
            market_m = mkt.get_market_data(t)
            macro_m.vix = market_m.vix
            r = valuation_engine.estimate(stmts_m, market_m, macro_m)
            rows_m.append({
                "Ticker":"  "+ t,"Sector":sector,
                "Price":market_m.current_price,
                "Fair Value":r.fair_value_base,
                "MoS %":round(r.margin_of_safety * 100, 1),
                "Confidence":r.confidence_score,
            })
        except Exception as e:
            print(f"  {t}: {e}")
    df_screen = pd.DataFrame(rows_m)
    df_screen["Ticker"] = df_screen["Ticker"].str.strip()

# ── Attach sentiment for every ticker ────────────────────────────────────────
print("\nAttaching sentiment scores (parallel, from cache if available)...\n")
all_sent_scores: dict = {}
all_unc_scores:  dict = {}

def _get_sent2(t):
    s = nlp.get_sentiment(t)
    return t, s.sentiment_score, s.uncertainty_score

with ThreadPoolExecutor(max_workers=6) as pool:
    futures2 = [pool.submit(_get_sent2, t) for t in df_screen["Ticker"].unique()]
    for fut in as_completed(futures2):
        t, sc, uc = fut.result()
        all_sent_scores[t] = sc
        all_unc_scores[t]  = uc
        print(f"  {t}: {sc:+.3f}")

df_screen["Sent Score"]  = df_screen["Ticker"].map(all_sent_scores)
df_screen["Uncertainty"] = df_screen["Ticker"].map(all_unc_scores)
df_screen["MoS norm"]    = np.tanh(df_screen["MoS %"] / 50)
df_screen["Composite"]   = (
    0.60 * df_screen["MoS norm"] + 0.40 * df_screen["Sent Score"]
).round(3)

print(f"\nReady — {df_screen['Sent Score'].notna().sum()}/{len(df_screen)} tickers with sentiment.")

In [ ]:
fig = px.scatter(
    df_screen.dropna(subset=["Sent Score"]),
    x="Sent Score",
    y="MoS %",
    color="Sector",
    size="Confidence",
    size_max=28,
    text="Ticker",
    hover_data=["Price", "Fair Value", "Confidence", "Uncertainty"],
    title="Valuation × Sentiment Quadrant  (bubble size = confidence score)",
    color_discrete_sequence=px.colors.qualitative.Bold,
)

fig.add_hline(y= 15, line_dash="dash", line_color="#4CAF50", line_width=1.5,
              annotation_text="MoS +15%", annotation_position="right")
fig.add_hline(y=-15, line_dash="dash", line_color="#F44336", line_width=1.5,
              annotation_text="MoS −15%", annotation_position="right")
fig.add_vline(x=  0, line_dash="dash", line_color="#9E9E9E", line_width=1.5)

for ax, ay, txt, bg in [
    ( 0.42,  0.83, "★ BUY ZONE<br>cheap + bullish",       "rgba(76,175,80,0.12)"),
    (-0.42,  0.83, "⚡ VALUE TRAP?<br>cheap + bearish",    "rgba(255,152,0,0.12)"),
    ( 0.42,  0.17, "⚠ MOMENTUM TRAP<br>expensive+bullish","rgba(255,87,34,0.12)"),
    (-0.42,  0.17, "▼ CLEAR AVOID<br>expensive+bearish",  "rgba(244,67,54,0.12)"),
]:
    fig.add_annotation(
        x=ax, y=ay, xref="x domain", yref="y domain",
        text=txt, showarrow=False,
        font=dict(size=10), bgcolor=bg, borderpad=4,
    )

fig.update_traces(textposition="top center")
fig.update_layout(
    height=640,
    xaxis_title="Sentiment Score  (← bearish … bullish →)",
    yaxis_title="Margin of Safety %",
)
fig.show()

## 6 · High-Conviction Watchlist

**Composite score** = 0.60 × tanh(MoS% / 50) + 0.40 × sentiment_score

- `tanh` normalises MoS to (−1, +1) and compresses extreme outliers
- 60 % fundamental weight, 40 % sentiment weight

**Entry criteria for high-conviction longs:**
- MoS > +15 % (stock is materially below estimated fair value)
- Confidence ≥ 0.50 (data complete, models agree)
- Sentiment score > 0 (at least neutral news flow)
- `composite > 0.30`

**Avoid criteria:**
- MoS < −15 %, confidence ≥ 0.45, composite < −0.30

In [ ]:
def _signal(row):
    if row["MoS %"] > 15 and row["Sent Score"] > 0.1:
        return "★ STRONG BUY"
    elif row["MoS %"] > 15 and row["Sent Score"] < -0.1:
        return "⚡ VALUE TRAP?"
    elif row["MoS %"] < -15 and row["Sent Score"] < -0.1:
        return "▼ CLEAR AVOID"
    elif row["MoS %"] < -15 and row["Sent Score"] > 0.1:
        return "⚠ MOMENTUM"
    else:
        return "~ NEUTRAL"

cols = ["Ticker", "Sector", "Price", "MoS %", "Confidence",
        "Sent Score", "Uncertainty", "Composite"]
df_rank = (
    df_screen[cols]
    .dropna()
    .sort_values("Composite", ascending=False)
    .reset_index(drop=True)
)
df_rank["Signal"] = df_rank.apply(_signal, axis=1)

print("=" * 74)
print("  COMPOSITE SIGNAL WATCHLIST  (60% valuation + 40% sentiment)")
print("=" * 74)
print(df_rank.to_string(index=False))

# ── High-conviction longs ─────────────────────────────────────────────────────
hc_long = df_rank[
    (df_rank["MoS %"]    > 15)
    & (df_rank["Confidence"] >= 0.50)
    & (df_rank["Sent Score"]  > 0.0)
    & (df_rank["Composite"]  > 0.30)
]

print(f"\n{'─'*74}")
print("  HIGH-CONVICTION LONGS  (MoS > 15%, conf ≥ 0.50, sent > 0, composite > 0.3)")
print(f"{'─'*74}")
if hc_long.empty:
    print("  None meet all criteria — consider loosening thresholds or expanding universe.")
else:
    for _, r in hc_long.iterrows():
        print(f"  {r['Ticker']:6s}  MoS={r['MoS %']:+.1f}%  conf={r['Confidence']:.2f}  "
              f"sent={r['Sent Score']:+.3f}  composite={r['Composite']:.3f}  {r['Signal']}")

# ── High-conviction avoids ────────────────────────────────────────────────────
hc_avoid = df_rank[
    (df_rank["MoS %"]    < -15)
    & (df_rank["Confidence"] >= 0.45)
    & (df_rank["Composite"]  < -0.30)
]

print(f"\n{'─'*74}")
print("  HIGH-CONVICTION AVOIDS  (MoS < −15%, conf ≥ 0.45, composite < −0.3)")
print(f"{'─'*74}")
if hc_avoid.empty:
    print("  None meet all criteria.")
else:
    for _, r in hc_avoid.iterrows():
        print(f"  {r['Ticker']:6s}  MoS={r['MoS %']:+.1f}%  conf={r['Confidence']:.2f}  "
              f"sent={r['Sent Score']:+.3f}  composite={r['Composite']:.3f}  {r['Signal']}")

print()
print("  Tip: Cross-check any long candidate with the DCF sensitivity heatmap")
print("       (notebook 02, section 3) to verify the upside holds under stress.")
print("  Tip: Guidance-raised stocks with composite > 0.4 are the rarest and")
print("       most reliable signal — earnings revision cycles tend to persist.")